In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 设置参数
SNR_DB = 25  # 目标信噪比(dB)
SEED = 42  # 随机种子
np.random.seed(SEED)

# 生成输出文件名
output_filename = f'Geerdataset_SNR{SNR_DB}.h5'
print(f"Output filename: {output_filename}")

# 1. 读取原始数据
print("Reading original data...")
with h5py.File('Geerdataset_bace.h5', 'r') as f:
    # 读取原始数据
    X_train = f['X_train'][:]  # shape: (2200, 1024)
    X_val = f['X_val'][:]      # shape: (2200, 1024)
    X_test = f['X_test'][:]    # shape: (2200, 1024)
    y_train = f['y_train'][:]  # shape: (2200,)
    y_val = f['y_val'][:]      # shape: (2200,)
    y_test = f['y_test'][:]    # shape: (2200,)
    
print(f"Data loaded: Train set {X_train.shape}, Val set {X_val.shape}, Test set {X_test.shape}")

# 2. 定义自适应噪声添加函数
def add_adaptive_noise(data, snr_db=SNR_DB):
    """
    添加自适应噪声
    snr_db: 目标信噪比(dB)
    data: shape (n_samples, 1024)
    """
    noisy_data = np.copy(data)
    n_samples = data.shape[0]
    signal_length = data.shape[1]
    
    for i in tqdm(range(n_samples), desc="Adding adaptive noise"):
        signal = data[i, :]
        signal_power = np.mean(signal ** 2)
        
        if signal_power < 1e-10:
            noisy_data[i] = data[i]
            continue
            
        snr_linear = 10 ** (snr_db / 10)
        noise_power = signal_power / snr_linear
        noise = np.random.normal(0, np.sqrt(noise_power), signal_length).astype(np.float32)
        noisy_signal = signal + noise
        noisy_data[i, :] = noisy_signal
    
    return noisy_data

# 3. 计算信噪比的函数
def calculate_snr(original, noisy):
    """
    计算实际信噪比
    """
    original = original.flatten()
    noisy = noisy.flatten()
    noise = noisy - original
    signal_power = np.mean(original ** 2)
    noise_power = np.mean(noise ** 2)
    
    if noise_power < 1e-10:
        return float('inf')
    
    snr_db = 10 * np.log10(signal_power / noise_power)
    return snr_db

def calculate_snr_for_dataset(original_data, noisy_data):
    """
    计算整个数据集的平均信噪比
    """
    snr_values = []
    
    for i in range(original_data.shape[0]):
        original_signal = original_data[i, :]
        noisy_signal = noisy_data[i, :]
        snr = calculate_snr(original_signal, noisy_signal)
        if snr != float('inf'):
            snr_values.append(snr)
    
    if snr_values:
        return np.mean(snr_values), np.std(snr_values), snr_values
    else:
        return float('inf'), 0, []

# 4. 只在测试集添加噪声
print("\nAdding noise to test set only...")
X_test_noisy = add_adaptive_noise(X_test, snr_db=SNR_DB)

# 5. 计算测试集的实际信噪比
print("\nCalculating actual SNR for test set...")
test_snr_mean, test_snr_std, test_snr_values = calculate_snr_for_dataset(X_test, X_test_noisy)
print(f"Test set actual average SNR: {test_snr_mean:.2f} ± {test_snr_std:.2f} dB")

# 6. 可视化信噪比分布
def plot_snr_distribution(snr_values, title, filename):
    """绘制信噪比分布图"""
    plt.figure(figsize=(10, 6))
    
    plt.hist(snr_values, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    
    mean_snr = np.mean(snr_values)
    plt.axvline(mean_snr, color='red', linestyle='dashed', linewidth=2, 
                label=f'Mean: {mean_snr:.2f} dB')
    
    plt.axvline(SNR_DB, color='green', linestyle='dashed', linewidth=2, 
                label=f'Target SNR: {SNR_DB} dB')
    
    plt.xlabel('SNR (dB)')
    plt.ylabel('Number of samples')
    plt.title(f'{title} - SNR Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

# 绘制测试集的信噪比分布
snr_distribution_filename = f'snr_distribution_SNR{SNR_DB}.png'
plot_snr_distribution(test_snr_values, f'Test Set (Target SNR={SNR_DB}dB)', snr_distribution_filename)

# 7. 保存数据集
def save_h5_dataset(filename, X_train, X_val, X_test, y_train, y_val, y_test):
    """保存数据集到HDF5文件"""
    with h5py.File(filename, 'w') as f:
        f.create_dataset('X_train', data=X_train, dtype=np.float32)
        f.create_dataset('X_val', data=X_val, dtype=np.float32)
        f.create_dataset('X_test', data=X_test, dtype=np.float32)
        f.create_dataset('y_train', data=y_train, dtype=np.int64)
        f.create_dataset('y_val', data=y_val, dtype=np.int64)
        f.create_dataset('y_test', data=y_test, dtype=np.int64)
    print(f"Saved: {filename}")

print("\nSaving dataset...")
save_h5_dataset(output_filename, 
                X_train, X_val, X_test_noisy,
                y_train, y_val, y_test)

# 8. 可视化函数 - 用于单个样本对比
def plot_single_sample_comparison(original, noisy, title, filename):
    """绘制单个样本的原始信号和噪声信号对比"""
    snr = calculate_snr(original, noisy)
    
    plt.figure(figsize=(12, 8))
    
    plt.subplot(3, 1, 1)
    plt.plot(original)
    plt.title(f"Original Signal - {title}")
    plt.xlabel("Time Points")
    plt.ylabel("Amplitude")
    
    plt.subplot(3, 1, 2)
    plt.plot(noisy)
    plt.title(f"Noisy Signal (Actual SNR={snr:.2f}dB, Target SNR={SNR_DB}dB) - {title}")
    plt.xlabel("Time Points")
    plt.ylabel("Amplitude")
    
    plt.subplot(3, 1, 3)
    noise_component = noisy - original
    plt.plot(noise_component)
    plt.title(f"Noise Component - {title}")
    plt.xlabel("Time Points")
    plt.ylabel("Amplitude")
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

# 9. 对测试集进行可视化对比
print("\nCreating visualization for test set...")
sample_idx = np.random.randint(0, 2200)

# 测试集样本对比
signal_comparison_filename = f'signal_comparison_SNR{SNR_DB}.png'
plot_single_sample_comparison(
    X_test[sample_idx, :].flatten(),
    X_test_noisy[sample_idx, :].flatten(),
    f"Test Sample (Target SNR={SNR_DB}dB)",
    signal_comparison_filename
)

# 10. 创建综合对比图
print("\nCreating comprehensive comparison...")
def create_comprehensive_comparison():
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 3, 1)
    plt.plot(X_test[sample_idx, :].flatten())
    plt.title("Original Test Sample")
    plt.xlabel("Time Points")
    plt.ylabel("Amplitude")
    
    plt.subplot(2, 3, 2)
    plt.plot(X_test_noisy[sample_idx, :].flatten())
    plt.title(f"Noisy Test Sample (Target SNR={SNR_DB}dB)")
    plt.xlabel("Time Points")
    
    plt.subplot(2, 3, 3)
    noise = X_test_noisy[sample_idx, :].flatten() - X_test[sample_idx, :].flatten()
    plt.plot(noise)
    plt.title("Noise Component")
    plt.xlabel("Time Points")
    
    plt.subplot(2, 3, 4)
    plt.hist(test_snr_values, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    mean_snr = np.mean(test_snr_values)
    plt.axvline(mean_snr, color='red', linestyle='dashed', linewidth=2, 
                label=f'Mean: {mean_snr:.2f} dB')
    plt.axvline(SNR_DB, color='green', linestyle='dashed', linewidth=2, 
                label=f'Target SNR: {SNR_DB} dB')
    plt.xlabel('SNR (dB)')
    plt.ylabel('Number of samples')
    plt.title(f'Test Set SNR Distribution (Target SNR={SNR_DB}dB)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 5)
    f_orig, Pxx_orig = plt.psd(X_test[sample_idx, :].flatten(), Fs=12000, visible=False)
    f_noisy, Pxx_noisy = plt.psd(X_test_noisy[sample_idx, :].flatten(), Fs=12000, visible=False)
    plt.cla()
    plt.semilogy(f_orig, Pxx_orig, label='Original Signal')
    plt.semilogy(f_noisy, Pxx_noisy, label='Noisy Signal')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('PSD (V²/Hz)')
    plt.title('Power Spectral Density Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 6)
    plt.axis('off')
    stats_text = f"""
    Signal Statistics (Target SNR={SNR_DB}dB):
    - Target SNR: {SNR_DB} dB
    - Actual Mean SNR: {test_snr_mean:.2f} dB
    - SNR Std Dev: {test_snr_std:.2f} dB
    - SNR Range: {np.min(test_snr_values):.2f} - {np.max(test_snr_values):.2f} dB
    - Total Samples: {len(test_snr_values)}
    """
    plt.text(0.1, 0.5, stats_text, fontsize=10, verticalalignment='center')
    plt.title('Statistics')
    
    plt.tight_layout()
    comprehensive_filename = f'comprehensive_comparison_SNR{SNR_DB}.png'
    plt.savefig(comprehensive_filename, dpi=300)
    plt.close()
    print(f"Comprehensive comparison saved: {comprehensive_filename}")

create_comprehensive_comparison()

# 11. 验证文件内容
def verify_h5_file(filename):
    print(f"\nVerifying file: {filename}")
    with h5py.File(filename, 'r') as f:
        print(f"X_train shape: {f['X_train'].shape}")
        print(f"X_test shape: {f['X_test'].shape}")
        print(f"y_train shape: {f['y_train'].shape}")
        print(f"y_test shape: {f['y_test'].shape}")
        
        sample_idx = np.random.randint(0, 2200)
        noisy_signal = f['X_test'][sample_idx, :]
        
        with h5py.File('Geerdataset_bace.h5', 'r') as orig_f:
            orig_signal_clean = orig_f['X_test'][sample_idx, :]
        
        noise = noisy_signal - orig_signal_clean
        signal_power = np.mean(orig_signal_clean ** 2)
        noise_power = np.mean(noise ** 2)
        
        if noise_power > 1e-10:
            snr = 10 * np.log10(signal_power / noise_power)
            print(f"Test sample {sample_idx} actual SNR: {snr:.2f} dB")
        else:
            print("Noise power too low to calculate SNR")

print("\nVerifying generated file...")
verify_h5_file(output_filename)

# 12. 打印信噪比统计
print(f"\nSNR Statistics:")
print(f"Test set actual average SNR: {test_snr_mean:.2f} ± {test_snr_std:.2f} dB")
print(f"Target SNR: {SNR_DB} dB")
print(f"Test set SNR range: {np.min(test_snr_values):.2f} - {np.max(test_snr_values):.2f} dB")

# 13. 保存信噪比统计数据到文件
snr_stats_filename = f'snr_statistics_SNR{SNR_DB}.txt'
with open(snr_stats_filename, 'w') as f:
    f.write("Test Set SNR Statistics Report\n")
    f.write("=" * 50 + "\n")
    f.write(f"Target SNR: {SNR_DB} dB\n")
    f.write(f"Actual Mean SNR: {test_snr_mean:.2f} ± {test_snr_std:.2f} dB\n")
    f.write(f"SNR Range: {np.min(test_snr_values):.2f} - {np.max(test_snr_values):.2f} dB\n")
    f.write(f"Total Samples: {len(test_snr_values)}\n")
    f.write("\n")
    f.write("Detailed SNR Distribution (first 20 samples):\n")
    for i, snr in enumerate(test_snr_values[:20]):
        f.write(f"  Sample {i}: {snr:.2f} dB\n")
    f.write("...\n")

print(f"\nAll operations completed! Saved files:")
print(f"1. Noisy dataset: {output_filename}")
print(f"2. SNR distribution: {snr_distribution_filename}")
print(f"3. Signal comparison: {signal_comparison_filename}")
print(f"4. Comprehensive comparison: comprehensive_comparison_SNR{SNR_DB}.png")
print(f"5. SNR statistics: {snr_stats_filename}")

Output filename: Geerdataset_SNR25.h5
Reading original data...
Data loaded: Train set (2200, 1024), Val set (2200, 1024), Test set (2200, 1024)

Adding noise to test set only...


Adding adaptive noise: 100%|██████████| 2200/2200 [00:00<00:00, 13450.51it/s]



Calculating actual SNR for test set...
Test set actual average SNR: 25.00 ± 0.19 dB

Saving dataset...
Saved: Geerdataset_SNR25.h5

Creating visualization for test set...

Creating comprehensive comparison...


C:\Users\11056\AppData\Local\Temp\ipykernel_48848\2074916562.py:227: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  plt.semilogy(f_orig, Pxx_orig, label='Original Signal')


Comprehensive comparison saved: comprehensive_comparison_SNR25.png

Verifying generated file...

Verifying file: Geerdataset_SNR25.h5
X_train shape: (2200, 1024)
X_test shape: (2200, 1024)
y_train shape: (2200,)
y_test shape: (2200,)
Test sample 702 actual SNR: 25.43 dB

SNR Statistics:
Test set actual average SNR: 25.00 ± 0.19 dB
Target SNR: 25 dB
Test set SNR range: 24.17 - 25.66 dB

All operations completed! Saved files:
1. Noisy dataset: Geerdataset_SNR25.h5
2. SNR distribution: snr_distribution_SNR25.png
3. Signal comparison: signal_comparison_SNR25.png
4. Comprehensive comparison: comprehensive_comparison_SNR25.png
5. SNR statistics: snr_statistics_SNR25.txt
